# GPU benchmark: fraud-tool-planner-llm

Runs the exact same manual-decode-loop generation as the local CPU
benchmarks in `engine/benchmark.py`, but on whatever GPU Colab hands out
(usually a T4 on the free tier), to get a real number for how much a GPU
actually helps -- rather than an estimate.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone --depth 1 https://github.com/Hemanth-hexo/Fraud_tool_LLM.git repo
%cd repo

In [ ]:
# Colab's preinstalled torch is already CUDA-enabled -- don't touch it, just
# add what's missing on top.
!pip install -q transformers peft accelerate safetensors sentencepiece

In [ ]:
import sys, json, time
sys.path.insert(0, "train")
sys.path.insert(0, "engine")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from prompt import build_messages
from generate import ManualGenerator
from constrained import ConstrainedToolPlanDecoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "Hemanth-hexo/fraud-tool-planner-lora"  # pushed to HF Hub from the local project

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to(DEVICE)
model = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
model.eval()

vocab_size = model.get_output_embeddings().weight.shape[0]
print("model loaded")

In [ ]:
rows = [json.loads(l) for l in open("data/eval.jsonl")][:30]
prompts = [
    tokenizer.apply_chat_template(build_messages(r["features"]), tokenize=False, add_generation_prompt=True)
    for r in rows
]

generator = ManualGenerator(model, tokenizer)

n_exact = 0
tokens_per_sec = []
total_wall_time = 0.0

for row, prompt_text in zip(rows, prompts):
    prompt_len = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])
    decoder = ConstrainedToolPlanDecoder(tokenizer, prompt_len, vocab_size, max_new_tokens=80)

    t0 = time.perf_counter()
    result = generator.generate(prompt_text, max_new_tokens=80, logits_processor=decoder)
    total_wall_time += time.perf_counter() - t0

    if result.decode_seconds:
        tokens_per_sec.append(result.tokens_per_second)

    parsed = json.loads(result.text)
    if set(parsed.get("selectedTools", [])) == set(row["output"]["selectedTools"]):
        n_exact += 1

avg_tps = sum(tokens_per_sec) / len(tokens_per_sec)
avg_latency_ms = 1000 * total_wall_time / len(rows)

print(f"device: {DEVICE}")
print(f"exact-match accuracy: {n_exact}/{len(rows)} ({100*n_exact/len(rows):.1f}%)")
print(f"avg tokens/sec: {avg_tps:.1f}")
print(f"avg latency per request: {avg_latency_ms:.0f}ms")
print()
print("For comparison, the CPU-only EC2 deployment measured ~10-14 tok/s and ~25-30s per request.")
print(f"GPU speedup: {avg_tps / 12:.1f}x tokens/sec (using ~12 tok/s as the CPU baseline)")